# LeetCode 2423 — Remove Letter To Equalize Frequency bằng Python

Link đề: https://leetcode.com/problems/remove-letter-to-equalize-frequency/ · Độ khó: **Easy** · Chủ đề: Hash Table, String, Counting

Skeleton LeetCode:

```python
class Solution:
    def equalFrequency(self, word: str) -> bool:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu bài toán: **xóa đúng một ký tự** khỏi chuỗi sao cho **mọi chữ cái còn lại** có cùng số lần xuất hiện.
2. Biết đếm tần suất bằng `Counter`, và đếm **"tần suất của tần suất"** (`Counter(cnt.values())`).
3. Hiểu chi tiết quan trọng: khi một chữ cái bị xóa hết (đếm về `0`) thì nó **không còn tính** là chữ cái còn lại.
4. Viết 4 cách: thử xóa từng vị trí rồi kiểm tra (brute), giảm đếm từng chữ cái, phân tích theo trường hợp bằng "tần suất của tần suất", và sắp xếp tần suất rồi thử hai ứng viên; đối chiếu bằng test ngẫu nhiên.
5. Tránh các lỗi: coi chuỗi đã đều là đúng (mà không xóa), giữ lại chữ cái đã về `0`, chỉ thử xóa từ chữ cái nhiều nhất, và chỉ đếm số loại tần suất.

## 2. Đề bài & ví dụ

Cho chuỗi `word` chỉ gồm chữ cái thường, đánh chỉ số từ 0. Bạn phải **chọn một chỉ số và xóa đúng một ký tự** tại đó, sao cho **mọi chữ cái còn xuất hiện trong chuỗi còn lại đều có cùng số lần**. Trả về `true` nếu làm được, ngược lại `false`.

Lưu ý:

- Số lần xuất hiện của một chữ cái đã bị xóa hết thì **bằng `0`** và chữ cái đó coi như **không còn tồn tại**; nó không vi phạm điều kiện.
- Bạn **bắt buộc** phải xóa đúng một ký tự (không được xóa 0 ký tự).

| Ví dụ | word | Output | Giải thích |
|---|---|---|---|
| 1 | `"abcc"` | `true` | xóa một `c` được `"abc"`, mỗi chữ cái xuất hiện 1 lần |
| 2 | `"aazz"` | `false` | xóa một `a` được `"azz"` (1 và 2); xóa một `z` được `"aaz"` (2 và 1); không có cách nào, và không được để nguyên |

Constraints: `2 <= word.length <= 100`; `word` chỉ gồm chữ cái tiếng Anh thường.

In [1]:
from collections import Counter

for word in ("abcc", "aazz"):
    print(word, Counter(word))
    for i in range(len(word)):
        rest = word[:i] + word[i + 1:]
        c = Counter(rest)
        print(f"  xóa chỉ số {i} -> {rest!r:6} đếm={dict(c)} đều nhau? {len(set(c.values())) == 1}")

abcc Counter({'c': 2, 'a': 1, 'b': 1})
  xóa chỉ số 0 -> 'bcc'  đếm={'b': 1, 'c': 2} đều nhau? False
  xóa chỉ số 1 -> 'acc'  đếm={'a': 1, 'c': 2} đều nhau? False
  xóa chỉ số 2 -> 'abc'  đếm={'a': 1, 'b': 1, 'c': 1} đều nhau? True
  xóa chỉ số 3 -> 'abc'  đếm={'a': 1, 'b': 1, 'c': 1} đều nhau? True
aazz Counter({'a': 2, 'z': 2})
  xóa chỉ số 0 -> 'azz'  đếm={'a': 1, 'z': 2} đều nhau? False
  xóa chỉ số 1 -> 'azz'  đếm={'a': 1, 'z': 2} đều nhau? False
  xóa chỉ số 2 -> 'aaz'  đếm={'a': 2, 'z': 1} đều nhau? False
  xóa chỉ số 3 -> 'aaz'  đếm={'a': 2, 'z': 1} đều nhau? False


## 3. Kiến thức nền

### 3.1. `Counter` và `.values()`

`Counter(word)` đếm từng ký tự. `c.values()` là các số lần xuất hiện (mỗi chữ cái một số). `set(c.values())` là tập các **mức tần suất khác nhau**: nếu độ dài tập này là `1` thì mọi chữ cái có cùng tần suất.

In [2]:
from collections import Counter
c = Counter("aabbcc")
print(c, list(c.values()), set(c.values()), len(set(c.values())) == 1)
c = Counter("aabbc")
print(c, set(c.values()), len(set(c.values())) == 1)

Counter({'a': 2, 'b': 2, 'c': 2}) [2, 2, 2] {2} True
Counter({'a': 2, 'b': 2, 'c': 1}) {1, 2} False


### 3.2. Cắt chuỗi để xóa một ký tự

Chuỗi bất biến (immutable) nên "xóa" nghĩa là tạo chuỗi mới: `word[:i] + word[i+1:]` (bỏ ký tự ở chỉ số `i`).

In [3]:
word = "abcc"
for i in range(len(word)):
    print(i, word[:i] + word[i + 1:])

0 bcc
1 acc
2 abc
3 abc


### 3.3. Giảm đếm và xóa khóa khi về `0`

Trên `Counter`, ta có thể `c[ch] -= 1`. Nhưng nếu `c[ch]` về `0` thì khóa vẫn còn trong `Counter` (`c.values()` vẫn chứa số `0`). Phải **xóa khóa** (`del c[ch]`) hoặc lọc các giá trị khác `0`; nếu không, số `0` sẽ bị tính như một mức tần suất.

In [4]:
from collections import Counter
c = Counter("ab")
c["a"] -= 1
print(c, list(c.values()))            # a vẫn còn với giá trị 0!
print([v for v in c.values() if v])   # lọc giá trị 0
del c["a"]
print(c)

Counter({'b': 1, 'a': 0}) [0, 1]
[1]
Counter({'b': 1})


### 3.4. "Tần suất của tần suất"

`fc = Counter(cnt.values())` cho biết **có bao nhiêu chữ cái** có mỗi mức tần suất. Ví dụ `"aabbbcc"`: `cnt = {a:2, b:3, c:2}`, `fc = {2: 2, 3: 1}`: hai chữ cái xuất hiện 2 lần, một chữ cái xuất hiện 3 lần. Cấu trúc này giúp phân tích theo trường hợp rất gọn.

In [5]:
from collections import Counter
cnt = Counter("aabbbcc")
fc = Counter(cnt.values())
print(cnt)
print(fc)
print(sorted(fc.items()))

Counter({'b': 3, 'a': 2, 'c': 2})
Counter({2: 2, 3: 1})
[(2, 2), (3, 1)]


### 3.5. `all(...)` và `any(...)`

`all(cond for ...)` đúng khi mọi điều kiện đúng; `any(...)` đúng khi có ít nhất một điều kiện đúng. Chúng dừng sớm khi biết kết quả.

In [6]:
print(all(x > 0 for x in [1, 2, 3]), all(x > 1 for x in [1, 2, 3]))
print(any(x > 2 for x in [1, 2, 3]), any(x > 3 for x in [1, 2, 3]))
print(all(x > 0 for x in []), any(x > 0 for x in []))   # tập rỗng

True False
True False
True False


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Tính tay vài ví dụ bằng cách xét tần suất.

- `"abcc"`: tần suất `[1, 1, 2]`. Giảm `2 → 1` thành `[1, 1, 1]`, đều ⇒ **true**.
- `"aazz"`: tần suất `[2, 2]`. Giảm một số thành `[1, 2]` ⇒ không đều ⇒ **false**.
- `"ab"`: tần suất `[1, 1]`. Xóa `a` còn `[1]` (b), một chữ cái duy nhất ⇒ đều ⇒ **true**. (Chữ cái `a` có đếm `0` thì coi như không còn.)
- `"abbcc"`: tần suất `[1, 2, 2]`. Giảm số `1` về `0` (chữ cái biến mất) còn `[2, 2]` ⇒ đều ⇒ **true**.
- `"aabbbcc"`: tần suất `[2, 3, 2]` → giảm `3 → 2` thành `[2, 2, 2]` ⇒ **true**.

Quy luật: ta chỉ **giảm đúng một tần suất đi 1**, sau đó các tần suất khác 0 phải bằng nhau. Số cách chọn chỉ bằng số chữ cái khác nhau (tối đa 26).

In [7]:
from collections import Counter

def freqs_after_decrement(freqs, idx):
    f = list(freqs)
    f[idx] -= 1
    return [x for x in f if x > 0]

for word in ("abcc", "aazz", "ab", "abbcc", "aabbbcc"):
    freqs = list(Counter(word).values())
    ok = any(len(set(freqs_after_decrement(freqs, i))) == 1 for i in range(len(freqs)))
    print(f"{word:8} tần suất {freqs} -> {ok}")

abcc     tần suất [1, 1, 2] -> True
aazz     tần suất [2, 2] -> False
ab       tần suất [1, 1] -> True
abbcc    tần suất [1, 2, 2] -> True
aabbbcc  tần suất [2, 3, 2] -> True


## 5. Cách 1 — Brute Force / baseline: thử xóa từng vị trí

**Ý tưởng:** với mỗi chỉ số `i`, tạo chuỗi mới sau khi xóa, đếm lại tần suất và kiểm tra mọi tần suất bằng nhau (`len(set(values)) == 1`). Nếu có một chỉ số thỏa thì trả `True`.

In [8]:
from typing import List
from collections import Counter

class SolutionBrute:
    def equalFrequency(self, word: str) -> bool:
        for i in range(len(word)):
            rest = word[:i] + word[i + 1:]
            c = Counter(rest)
            if len(set(c.values())) == 1:
                return True
        return False

In [9]:
sol = SolutionBrute()
print(sol.equalFrequency("abcc"))   # True
print(sol.equalFrequency("aazz"))   # False
print(sol.equalFrequency("ab"))     # True

True
False
True


**Độ phức tạp:** mỗi chỉ số tạo chuỗi `O(n)` và đếm `O(n)`, có `n` chỉ số ⇒ Time `O(n^2)` với `n = |word| <= 100` (chỉ ~10^4 thao tác). Space `O(n)`. **Nút thắt:** ta đếm lại cả chuỗi dù chỉ thay đổi một ký tự; vì chỉ có tối đa 26 loại chữ cái, ta chỉ cần thử "giảm một chữ cái" trên bảng đếm.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước đột phá:** việc xóa một ký tự chỉ làm **một mức đếm giảm đi 1**. Có tối đa 26 chữ cái, nên chỉ cần thử giảm từng chữ cái một:

- **Invariant:** bảng đếm `cnt` của `word` đã tính; mỗi thử nghiệm thay đổi `cnt[ch]` đúng `-1`.
- **Quy tắc:** với mỗi chữ cái `ch`: giảm `cnt[ch]`, bỏ các giá trị `0`, **nếu** các giá trị còn lại đều bằng nhau (hoặc không còn giá trị nào) **thì** trả `True`; rồi khôi phục `cnt[ch]`.
- Chữ cái bị giảm về `0` chính là trường hợp "xóa chữ cái duy nhất xuất hiện 1 lần".

Cách còn nhanh hơn: phân tích "tần suất của tần suất" `fc` để trả lời trực tiếp bằng vài trường hợp (xem mục 7).

## 7. Các cách tiếp theo

### Cách 2 — Giảm đếm từng chữ cái (tối đa 26 thử nghiệm)

Giữ một `Counter`, với mỗi chữ cái giảm 1, lọc bỏ `0`, rồi kiểm tra.

In [10]:
from typing import List
from collections import Counter

class SolutionDecrement:
    def equalFrequency(self, word: str) -> bool:
        cnt = Counter(word)
        for ch in list(cnt):
            cnt[ch] -= 1
            remaining = [v for v in cnt.values() if v > 0]    # bỏ chữ cái đã về 0
            if len(set(remaining)) <= 1:
                return True
            cnt[ch] += 1                                      # khôi phục
        return False

In [11]:
print(SolutionDecrement().equalFrequency("abcc"))   # True
print(SolutionDecrement().equalFrequency("aazz"))   # False
print(SolutionDecrement().equalFrequency("ab"))     # True

True
False
True


**Độ phức tạp:** đếm `O(n)`, mỗi thử nghiệm duyệt tối đa 26 giá trị, tổng `O(n + 26·26)` ⇒ Time `O(n)`, Space `O(26) = O(1)`.

### Cách 3 — Phân tích theo trường hợp bằng "tần suất của tần suất"

Gọi `cnt` là đếm từng chữ cái và `fc = Counter(cnt.values())`. Có các trường hợp chỉ ra đáp án `True`:

1. **Chỉ có một chữ cái** (`len(cnt) == 1`): xóa một ký tự vẫn là một chữ cái duy nhất ⇒ `True`.
2. **Mọi chữ cái xuất hiện đúng 1 lần** (`fc` là `{1: k}`): xóa một chữ cái, các chữ còn lại vẫn là 1 ⇒ `True`.
3. **Đúng hai mức tần suất `lo < hi`:**
   - nếu `lo == 1` và chỉ có **một** chữ cái ở mức `lo` ⇒ xóa chữ cái đó ⇒ các chữ còn lại cùng `hi` ⇒ `True`;
   - nếu `hi == lo + 1` và chỉ có **một** chữ cái ở mức `hi` ⇒ xóa một ký tự của chữ cái đó ⇒ tất cả cùng `lo` ⇒ `True`.
4. Còn lại ⇒ `False` (ví dụ mọi chữ cái cùng tần suất `f >= 2` với nhiều chữ cái: `"aazz"`).

In [12]:
from typing import List
from collections import Counter

class SolutionCases:
    def equalFrequency(self, word: str) -> bool:
        cnt = Counter(word)
        if len(cnt) == 1:
            return True
        fc = Counter(cnt.values())
        if len(fc) == 1:
            return next(iter(fc)) == 1          # tất cả cùng tần suất: chỉ ổn khi đều là 1
        if len(fc) == 2:
            lo, hi = sorted(fc)
            if lo == 1 and fc[lo] == 1:
                return True
            if hi == lo + 1 and fc[hi] == 1:
                return True
        return False

In [13]:
print(SolutionCases().equalFrequency("abcc"))   # True
print(SolutionCases().equalFrequency("aazz"))   # False
print(SolutionCases().equalFrequency("ab"))     # True

True
False
True


**Độ phức tạp:** Time `O(n)` (đếm chuỗi, phần còn lại là hằng số vì tối đa 26 chữ cái), Space `O(1)`.

### Cách 4 — Sắp xếp tần suất và thử hai ứng viên

Sắp xếp danh sách tần suất. Vì các chữ cái có cùng tần suất là **hoán đổi được** với nhau, chỉ cần thử giảm phần tử **nhỏ nhất** (đầu danh sách) hoặc **lớn nhất** (cuối danh sách). Giảm một phần tử ở giữa (khác cả nhỏ nhất lẫn lớn nhất) không thể làm hai cực trị bằng nhau.

In [14]:
from typing import List
from collections import Counter

class SolutionSorted:
    def equalFrequency(self, word: str) -> bool:
        freqs = sorted(Counter(word).values())
        for idx in (0, len(freqs) - 1):
            f = freqs[:]
            f[idx] -= 1
            f = [x for x in f if x > 0]
            if len(set(f)) <= 1:
                return True
        return False

In [15]:
print(SolutionSorted().equalFrequency("abcc"))   # True
print(SolutionSorted().equalFrequency("aazz"))   # False
print(SolutionSorted().equalFrequency("ab"))     # True

True
False
True


**Độ phức tạp:** Time `O(n + 26 log 26)` = `O(n)`, Space `O(1)`.

## 8. TRACE từng bước

Cách **giảm đếm** với `word = "abcc"`: `cnt = {a:1, b:1, c:2}`.

| Thử | Giảm | `cnt` sau khi giảm | Giá trị `> 0` | Khác nhau | Kết quả |
|---|---|---|---|---|---|
| 1 | `a` | `{a:0, b:1, c:2}` | `[1, 2]` | 2 | không |
| 2 | `b` | `{a:1, b:0, c:2}` | `[1, 2]` | 2 | không |
| 3 | `c` | `{a:1, b:1, c:1}` | `[1, 1, 1]` | 1 | **True** |

In [16]:
from collections import Counter

def trace_decrement(word):
    cnt = Counter(word)
    print("đếm ban đầu:", dict(cnt))
    for ch in list(cnt):
        cnt[ch] -= 1
        remaining = [v for v in cnt.values() if v > 0]
        distinct = sorted(set(remaining))
        print(f"giảm {ch!r}: cnt={dict(cnt)}, còn lại={remaining}, mức khác nhau={distinct}")
        if len(distinct) <= 1:
            print("   -> các mức đều nhau => True")
            return True
        cnt[ch] += 1
    print("không có cách nào => False")
    return False

print(trace_decrement("abcc"))
print(trace_decrement("aazz"))

đếm ban đầu: {'a': 1, 'b': 1, 'c': 2}
giảm 'a': cnt={'a': 0, 'b': 1, 'c': 2}, còn lại=[1, 2], mức khác nhau=[1, 2]
giảm 'b': cnt={'a': 1, 'b': 0, 'c': 2}, còn lại=[1, 2], mức khác nhau=[1, 2]
giảm 'c': cnt={'a': 1, 'b': 1, 'c': 1}, còn lại=[1, 1, 1], mức khác nhau=[1]
   -> các mức đều nhau => True
True
đếm ban đầu: {'a': 2, 'z': 2}
giảm 'a': cnt={'a': 1, 'z': 2}, còn lại=[1, 2], mức khác nhau=[1, 2]
giảm 'z': cnt={'a': 2, 'z': 1}, còn lại=[2, 1], mức khác nhau=[1, 2]
không có cách nào => False
False


## 9. Code nộp LeetCode

Dùng cách giảm đếm (dễ kiểm chứng, độ phức tạp tốt):

In [17]:
from typing import List
from collections import Counter

class Solution:
    def equalFrequency(self, word: str) -> bool:
        cnt = Counter(word)
        for ch in list(cnt):
            cnt[ch] -= 1
            remaining = [v for v in cnt.values() if v > 0]
            if len(set(remaining)) <= 1:
                return True
            cnt[ch] += 1
        return False

In [18]:
print(Solution().equalFrequency("abcc"))
print(Solution().equalFrequency("aazz"))

True
False


## 10. Vì sao thuật toán đúng

1. **Hiệu ứng của một lần xóa.** Xóa một ký tự `ch` giảm `cnt[ch]` đúng 1 và không đổi các chữ cái khác. Hai chuỗi có cùng `cnt` sau khi xóa thì cùng kết quả (chỉ tần suất mới quan trọng, không phải vị trí).
2. **Phạm vi thử.** Mọi cách xóa đều tương ứng với một chữ cái `ch` có `cnt[ch] >= 1`. Có tối đa 26 chữ cái nên thử hết là đủ ⇒ không bỏ sót.
3. **Điều kiện đều.** "Mọi chữ cái còn lại cùng số lần" ⇔ tập các đếm `> 0` có không quá một giá trị khác nhau. Trường hợp tập rỗng không xảy ra vì `|word| >= 2` nên sau khi xóa còn ít nhất 1 ký tự; vẫn kiểm `<= 1` cho an toàn.
4. **Bỏ chữ cái về 0.** Chữ cái bị xóa hết không còn trong chuỗi nên không được tính; do đó phải lọc `v > 0`.
5. **Phân tích trường hợp (Cách 3).** Sau khi giảm một mức đếm, để các đếm còn lại bằng nhau ta cần: hoặc chỉ có một chữ cái, hoặc tất cả đều 1 (xóa bất kỳ), hoặc đúng hai mức mà mức thấp là 1 với một đại diện duy nhất (biến mất), hoặc mức cao hơn mức thấp đúng 1 với một đại diện duy nhất (hạ xuống). Các trường hợp khác có ít nhất hai chữ cái ở mức "lệch" nên một lần giảm không đủ.
6. **Cách 4.** Hai cực trị (nhỏ nhất, lớn nhất) quyết định tính đều; muốn đều thì phải thay đổi một trong hai, vì vậy thử giảm đầu và cuối danh sách đã sắp là đủ.

## 11. Phân tích Time / Space Complexity

Đặt `n = len(word)` và `A = 26` là kích thước bảng chữ cái.

| Cách | Time | Space |
|---|---|---|
| Brute (xóa từng vị trí) | `O(n^2)` | `O(n)` |
| Giảm đếm từng chữ cái | `O(n + A^2)` | `O(A)` |
| Phân tích theo trường hợp | `O(n + A)` | `O(A)` |
| Sắp xếp tần suất, thử 2 ứng viên | `O(n + A log A)` | `O(A)` |

Với `n <= 100` mọi cách đều tức thì; điểm cốt lõi là chuyển bài toán từ chuỗi sang **bảng tần suất cỡ 26**.

## 12. Bộ test

### 12.1. Ví dụ của đề và edge case

In [19]:
sols = [SolutionBrute(), SolutionDecrement(), SolutionCases(), SolutionSorted(), Solution()]
cases = [
    ("abcc", True),                  # ví dụ 1
    ("aazz", False),                 # ví dụ 2
    ("ab", True),                    # n = 2, hai chữ cái khác nhau
    ("aa", True),                    # n = 2, một chữ cái (còn lại "a")
    ("abc", True),                   # tất cả 1
    ("aabb", False),                 # đã đều nhưng phải xóa
    ("abbcc", True),                 # mức 1 biến mất
    ("aaabb", True),                 # hạ mức cao 3 -> 2
    ("aabbcc", False),               # đều f=2 với 3 chữ cái
    ("abcdd", True),
    ("abbccc", False),               # 1,2,3
    ("aabbbcc", True),               # 2,3,2
    ("aaaaaaaaaab", True),           # 10 và 1: xóa chữ b
    ("aabbbb", False),               # 2 và 4
    ("a" * 100, True),               # một chữ cái, n lớn nhất
    ("ab" * 50, False),              # 50,50
    ("abcdefghijklmnopqrstuvwxyz", True),   # 26 chữ cái, mỗi chữ 1 lần
    ("aabbbccc", False),             # 2,3,3 -> không đều sau một lần xóa
]
for word, want in cases:
    for s in sols:
        assert s.equalFrequency(word) == want, (type(s).__name__, word, s.equalFrequency(word), want)
print("tất cả test cố định đều đúng với cả 5 cách:", len(cases))

tất cả test cố định đều đúng với cả 5 cách: 18


### 12.2. Đối chiếu ngẫu nhiên (có seed)

Sinh chuỗi đúng constraints: độ dài `2..100`, chữ cái thường. Dùng bảng chữ cái nhỏ (nhiều chuỗi "gần đều") và bảng chữ cái lớn. Brute là chuẩn đối chiếu.

In [20]:
import random
import string
random.seed(2423)

def gen(alpha, lo, hi):
    n = random.randint(lo, hi)
    return "".join(random.choice(alpha) for _ in range(n))

n_true = n_false = 0
for _ in range(1500):
    alpha = random.choice(["ab", "abc", "abcd", string.ascii_lowercase])
    word = gen(alpha, 2, random.choice([5, 10, 30, 100]))
    want = sols[0].equalFrequency(word)
    for s in sols[1:]:
        assert s.equalFrequency(word) == want, (type(s).__name__, word)
    n_true += want
    n_false += (not want)
print("1500 chuỗi ngẫu nhiên: các cách khớp brute; True =", n_true, ", False =", n_false)

1500 chuỗi ngẫu nhiên: các cách khớp brute; True = 703 , False = 797


In [21]:
# Chuỗi cố ý "gần đều": mỗi chữ cái xuất hiện f lần, rồi thêm / bớt một ký tự
random.seed(24230)
cnt_true = cnt_false = 0
for _ in range(1000):
    k = random.randint(1, 6)
    f = random.randint(1, 10)
    letters = random.sample(string.ascii_lowercase, k)
    chars = [ch for ch in letters for _ in range(f)]
    mode = random.choice(["add", "remove", "none", "swap"])
    if mode == "add":
        chars.append(random.choice(string.ascii_lowercase))
    elif mode == "remove" and len(chars) > 2:
        chars.pop(random.randrange(len(chars)))
    elif mode == "swap" and len(chars) > 3:
        chars[random.randrange(len(chars))] = random.choice(string.ascii_lowercase)
    random.shuffle(chars)
    word = "".join(chars)
    if not (2 <= len(word) <= 100):
        continue
    want = sols[0].equalFrequency(word)
    for s in sols[1:]:
        assert s.equalFrequency(word) == want, (type(s).__name__, word)
    cnt_true += want
    cnt_false += (not want)
print("test gần đều khớp; True =", cnt_true, ", False =", cnt_false)
assert cnt_true > 50 and cnt_false > 50

test gần đều khớp; True = 474 , False = 511


### 12.3. Phản ví dụ: bắt buộc phải xóa

`"aabb"` đã đều (mỗi chữ 2 lần) nhưng đề bắt buộc xóa đúng một ký tự, nên kết quả `False`. Hãy kiểm tra tất cả chuỗi dạng "mỗi chữ `f >= 2` lần, ít nhất 2 chữ cái" đều cho `False`.

In [22]:
random.seed(99)
for _ in range(200):
    k = random.randint(2, 6)
    f = random.randint(2, 10)
    letters = random.sample(string.ascii_lowercase, k)
    word = "".join(ch * f for ch in letters)
    assert 2 <= len(word) <= 100
    for s in sols:
        assert s.equalFrequency(word) is False, (type(s).__name__, word)
print("mọi chuỗi đều với f >= 2 và k >= 2 chữ cái -> False")

mọi chuỗi đều với f >= 2 và k >= 2 chữ cái -> False


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Coi chuỗi đã đều là đúng (không xóa)

In [23]:
from collections import Counter

def sai_da_deu(word: str) -> bool:
    return len(set(Counter(word).values())) == 1       # SAI: đề bắt buộc xóa 1 ký tự

word = "aazz"
print("đúng:", Solution().equalFrequency(word), "| sai:", sai_da_deu(word))
assert Solution().equalFrequency(word) != sai_da_deu(word)

đúng: False | sai: True


**Vì sao sai:** `"aazz"` đã đều, nhưng đề **bắt buộc** xóa đúng một ký tự và sau đó vẫn phải đều; xóa một ký tự thì `a` và `z` lệch nhau (`1` và `2`).

### Lỗi 2: Giữ chữ cái đã về `0` như một mức tần suất

In [24]:
def sai_giu_so_0(word: str) -> bool:
    cnt = Counter(word)
    for ch in list(cnt):
        cnt[ch] -= 1
        if len(set(cnt.values())) == 1:                # SAI: không lọc giá trị 0
            return True
        cnt[ch] += 1
    return False

word = "ab"
print("đúng:", Solution().equalFrequency(word), "| sai:", sai_giu_so_0(word))
assert Solution().equalFrequency(word) != sai_giu_so_0(word)

đúng: True | sai: False


**Vì sao sai:** xóa `a` khỏi `"ab"` còn `"b"`: một chữ cái duy nhất ⇒ đều. Nhưng code sai thấy `{a: 0, b: 1}` có hai mức (`0` và `1`) nên kết luận `False`. Phải bỏ các giá trị `0` (hoặc `del cnt[ch]`).

### Lỗi 3: Chỉ thử giảm chữ cái xuất hiện nhiều nhất

In [25]:
def sai_chi_giam_max(word: str) -> bool:
    cnt = Counter(word)
    ch = max(cnt, key=lambda c: cnt[c])                # SAI: chỉ thử một chữ cái
    cnt[ch] -= 1
    return len(set(v for v in cnt.values() if v > 0)) <= 1

word = "abbcc"
print("đúng:", Solution().equalFrequency(word), "| sai:", sai_chi_giam_max(word))
assert Solution().equalFrequency(word) != sai_chi_giam_max(word)

đúng: True | sai: False


**Vì sao sai:** `"abbcc"` đúng khi xóa `a` (mức thấp nhất = 1) để còn `"bbcc"`, nhưng code sai chỉ thử giảm một chữ cái phổ biến (`b`, ra `[1, 1, 2]`). Phải thử mọi chữ cái (hoặc hai cực trị).

### Lỗi 4: Chỉ kiểm tra có nhiều nhất 2 mức tần suất

In [26]:
def sai_chi_dem_so_muc(word: str) -> bool:
    return len(set(Counter(word).values())) <= 2        # SAI: quá lỏng

word = "aabbbccc"
print("đúng:", Solution().equalFrequency(word), "| sai:", sai_chi_dem_so_muc(word))
assert Solution().equalFrequency(word) != sai_chi_dem_so_muc(word)

đúng: False | sai: True


**Vì sao sai:** `"aabbbccc"` có hai mức `{2, 3}` nhưng một lần xóa không làm ba chữ cái bằng nhau (xóa `a` → `1,3,3`; xóa `b` → `2,2,3`). Cần kiểm tra thêm: mức thấp là `1` với một đại diện, hoặc mức cao hơn đúng `1` với một đại diện.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Dữ liệu là chuỗi chữ cái thường ⇒ bảng đếm chỉ có tối đa 26 phần tử, nên "thử từng chữ cái" rất rẻ.
- Cụm "mọi chữ cái có cùng tần suất" ⇒ kiểm tra `len(set(cnt.values())) == 1`.
- Cụm "xóa đúng một ký tự" ⇒ giảm một đếm đúng `1`, đừng quên **bắt buộc** (không được để nguyên).
- Nhớ lọc đếm `0` hoặc dùng `del`.
- Khi chưa chắc về phân tích trường hợp, hãy viết brute rồi **so sánh ngẫu nhiên** với lời giải nhanh; ở bài này làm vậy bắt được nhiều lỗi nhỏ.
- Hãy tự kiểm với `"aa"` (n = 2, một chữ cái) và `"ab"` (hai chữ cái, đều 1).

## 15. Biến thể của bài

1. **Xóa tối đa một ký tự** (được phép không xóa): thêm điều kiện chuỗi đã đều.
2. **Thêm một ký tự** thay vì xóa: tương tự nhưng tăng một đếm.
3. **Xóa `k` ký tự:** bài toán khó hơn, cần quy hoạch hoặc tham lam dựa trên tần suất.
4. **Số ký tự tối thiểu cần xóa** để mọi chữ cái có cùng tần suất (xem "Minimum Deletions to Make Character Frequencies Unique" cùng ý tưởng đếm tần suất).
5. **Trả về chỉ số** của một ký tự cần xóa.
6. **Chuỗi Unicode / mảng số**: đổi `Counter` để đếm phần tử bất kỳ.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute | xóa từng vị trí, đếm lại, kiểm tra | `O(n^2)` | `O(n)` | đối chứng cho test nhỏ |
| 2. Giảm đếm | thử giảm từng chữ cái, lọc `0` | `O(n + A^2)` | `O(A)` | cách khuyến nghị: ngắn, chắc chắn đúng |
| 3. Theo trường hợp | phân tích `Counter(cnt.values())` | `O(n + A)` | `O(A)` | nhanh nhất, nhưng dễ sót trường hợp |
| 4. Sắp xếp tần suất | chỉ thử giảm cực nhỏ / cực lớn | `O(n + A log A)` | `O(A)` | khi muốn lập luận ngắn mà vẫn an toàn |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** `Solution().equalFrequency("zzzy")` là gì? Còn `"abcde"`?

**Bài 2 (tính tay).** Với `"aabbbcc"`, viết bảng đếm và chỉ ra cách xóa làm mọi chữ cái bằng nhau.

**Bài 3 (điền TODO).** Hoàn thành hàm `deu_nhau(cnt)` kiểm tra mọi giá trị khác `0` của `Counter` bằng nhau.

In [27]:
def deu_nhau(cnt):
    # TODO: trả về True nếu tất cả giá trị > 0 trong cnt bằng nhau (bỏ qua giá trị 0)
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi.

**Bài 4 (tự viết lại).** Viết `equal_or_one_removal(word)` trả về `True` nếu chuỗi đã đều **hoặc** có thể đều sau khi xóa đúng một ký tự (xóa tối đa 1).

**Bài 5 (biến thể / mini challenge).** Viết `min_deletions_equal(word)` trả về số ký tự ít nhất phải xóa để mọi chữ cái còn lại có cùng tần suất (được phép xóa hết một chữ cái). Gợi ý: thử mọi mức tần suất mục tiêu `t`, mỗi chữ cái có đếm `c >= t` giữ `t` (xóa `c - t`), có đếm `c < t` thì xóa cả `c`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `"zzzy"` có `z:3, y:1`; xóa `y` còn `zzz` nên `True`; `"abcde"` mỗi chữ 1 lần nên `True`.
- **Mức 2 (vừa):** Bài 3: `vals = [v for v in cnt.values() if v > 0]` rồi `len(set(vals)) <= 1`. Bài 4: trả `True` nếu `deu_nhau(Counter(word))`, nếu không thì dùng lời giải chính.
- **Mức 3 (sâu):** Bài 5: với mỗi `t` từ `1` đến `max(cnt.values())`: tổng xóa = `sum(c - t if c >= t else c for c in cnt.values())`; lấy `min`. Mức `t` thỏa khi mọi chữ cái còn lại đều có đúng `t`, những chữ cái có `c < t` bị xóa hết.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [28]:
from collections import Counter

# Bài 1
assert Solution().equalFrequency("zzzy") is True
assert Solution().equalFrequency("abcde") is True

# Bài 2: {a:2, b:3, c:2} -> giảm b về 2 -> True
assert Counter("aabbbcc") == {"a": 2, "b": 3, "c": 2}
assert Solution().equalFrequency("aabbbcc") is True

# Bài 3
def deu_nhau(cnt):
    vals = [v for v in cnt.values() if v > 0]
    return len(set(vals)) <= 1

# Bài 4
def equal_or_one_removal(word):
    return deu_nhau(Counter(word)) or Solution().equalFrequency(word)

# Bài 5
def min_deletions_equal(word):
    cnt = Counter(word)
    best = len(word)
    for t in range(1, max(cnt.values()) + 1):
        total = sum(c - t if c >= t else c for c in cnt.values())
        best = min(best, total)
    return best

assert deu_nhau(Counter("aabb")) is True
assert deu_nhau(Counter("aab")) is False
assert deu_nhau(Counter({"a": 0, "b": 3})) is True
assert equal_or_one_removal("aabb") is True
assert equal_or_one_removal("aazz") is True
assert equal_or_one_removal("aabbbbcc") is False
assert min_deletions_equal("aabb") == 0
assert min_deletions_equal("aab") == 1            # xóa b (hoặc a)
assert min_deletions_equal("aaabbc") == 2         # t=2: xóa 1 (a) + 0 (b) + 1 (c) = 2
assert min_deletions_equal("abcabc") == 0
assert min_deletions_equal("a") == 0
print("tất cả bài tập OK")

tất cả bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (thử giảm một chữ cái, lọc `0`, kiểm tra đều)

```python
class Solution:
    def equalFrequency(self, word: str) -> bool:
        cnt = Counter(word)
        for ch in list(cnt):
            cnt[ch] -= 1
            remaining = [v for v in cnt.values() if v > 0]
            if len(set(remaining)) <= 1:
                return True
            cnt[ch] += 1
        return False
```

**Checklist**

- [ ] Bắt buộc xóa **đúng một** ký tự (không trả `True` chỉ vì chuỗi đã đều)?
- [ ] Đã bỏ các đếm `0` (hoặc `del`) trước khi so sánh?
- [ ] Đã **khôi phục** `cnt[ch] += 1` sau mỗi lần thử?
- [ ] Đã thử mọi chữ cái, không chỉ chữ cái nhiều nhất?
- [ ] Đã thử `"aa"`, `"ab"`, `"aazz"` và chuỗi toàn một chữ cái dài 100?
- [ ] Nếu dùng phân tích trường hợp: đã đối chiếu với brute bằng test ngẫu nhiên?
- [ ] Signature đúng `equalFrequency(self, word: str) -> bool` và có `from typing import List` nếu cần?